## Dataset analysis notebook

Counts images per split and per class for both datasets:

- **KL Grading** (classification) — `kneeKL224`, 5 classes (`0..4`)
- **YOLOv8 Knee ROI** (YOLO-detector crop output) — `densenet121_yolo_square_roi_trainvaltest_v2`, organized by KL grade

Output is plain-text tables — easy to read in a Colab cell.

In [1]:
# ─── 1. Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ─── 2. Paths & labels ─────────────────────────────────────────────────────
from pathlib import Path
from collections import defaultdict
from PIL import Image

DATASETS_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData")

KL_ROOT   = DATASETS_ROOT / "ClsKLData" / "kneeKL224"
YOLO_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2")

SPLITS = ("train", "val", "test")
GRADES = (0, 1, 2, 3, 4)
GRADE_LABELS = {
    0: "Healthy",
    1: "Doubtful",
    2: "Minimal",
    3: "Moderate",
    4: "Severe",
}
IMG_EXTS = {".png", ".jpg", ".jpeg"}

print("KL  root :", KL_ROOT,  "exists?", KL_ROOT.exists())
print("YOLO root:", YOLO_ROOT, "exists?", YOLO_ROOT.exists())

KL  root : /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224 exists? True
YOLO root: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2 exists? True


In [3]:
# ─── 3. Counting helper ────────────────────────────────────────────────────
def count_dataset(root):
    """Walk root/{split}/{grade}/*.png and return counts[split][grade] = N images,
    plus image_stats (mode, extension, size) per split."""
    counts = {s: {g: 0 for g in GRADES} for s in SPLITS}
    image_stats = {s: {"modes": defaultdict(int),
                        "exts": defaultdict(int),
                        "sizes": set()} for s in SPLITS}
    for split in SPLITS:
        if not (root / split).exists():
            continue
        for grade in GRADES:
            grade_dir = root / split / str(grade)
            if not grade_dir.exists():
                continue
            for p in sorted(grade_dir.iterdir()):
                if p.suffix.lower() not in IMG_EXTS:
                    continue
                counts[split][grade] += 1
                try:
                    with Image.open(p) as im:
                        image_stats[split]["modes"][im.mode] += 1
                        image_stats[split]["exts"][p.suffix.lower()] += 1
                        image_stats[split]["sizes"].add(im.size)
                except Exception as e:
                    print(f"  [warn] cannot read {p}: {e}")
    return counts, image_stats


def print_table(title, counts, image_stats):
    """Pretty-print a per-dataset summary in plain text."""
    # Split totals & percentages
    split_totals = {s: sum(counts[s].values()) for s in SPLITS}
    grand_total  = sum(split_totals.values())

    print()
    print("=" * 78)
    print(f" {title}")
    print("=" * 78)

    # -------------------------------------------------------------------
    # 1. Per-split image count and percentage of dataset
    # -------------------------------------------------------------------
    print("\n[1] Split sizes")
    print(f"  {'split':<8} {'images':>8} {'% of total':>12}")
    print("  " + "-" * 32)
    if grand_total == 0:
        print("  (no images found)")
    else:
        for s in SPLITS:
            n = split_totals[s]
            if n == 0:
                print(f"  {s:<8} {n:>8} {0.0:>11.2f}%")
            else:
                pct = 100.0 * n / grand_total
                print(f"  {s:<8} {n:>8} {pct:>11.2f}%")
        print("  " + "-" * 32)
        print(f"  {'TOTAL':<8} {grand_total:>8}")

    # -------------------------------------------------------------------
    # 2. Split × Class matrix
    # -------------------------------------------------------------------
    print("\n[2] Split × Class matrix (images)")
    header_left = f"  {'split':<8}"
    header_cols = "".join(f"{f'g{g}':>8}" for g in GRADES)
    header_tot  = f"{'total':>8}"
    print(header_left + header_cols + header_tot)
    print("  " + "-" * (8 + 8 * len(GRADES) + 8))
    for s in SPLITS:
        if split_totals[s] == 0:
            continue
        cells = "".join(f"{counts[s][g]:>8}" for g in GRADES)
        print(f"  {s:<8}{cells}{split_totals[s]:>8}")
    print("  " + "-" * (8 + 8 * len(GRADES) + 8))
    # Per-column total
    col_totals = [sum(counts[s][g] for s in SPLITS) for g in GRADES]
    cells = "".join(f"{t:>8}" for t in col_totals)
    print(f"  {'TOTAL':<8}{cells}{grand_total:>8}")

    # -------------------------------------------------------------------
    # 3. Class label & per-split class percentage
    # -------------------------------------------------------------------
    if grand_total > 0:
        print("\n[3] Class labels and overall distribution")
        print(f"  {'grade':<6} {'label':<10} {'count':>8} {'% of total':>12}")
        print("  " + "-" * 40)
        for g in GRADES:
            n = col_totals[g]
            pct = 100.0 * n / grand_total if grand_total else 0.0
            print(f"  g{g:<5} {GRADE_LABELS[g]:<10} {n:>8} {pct:>11.2f}%")

    # -------------------------------------------------------------------
    # 4. Image properties (mode, extension, size)
    # -------------------------------------------------------------------
    print("\n[4] Image properties")
    for s in SPLITS:
        if split_totals[s] == 0:
            continue
        n_imgs = sum(image_stats[s]["modes"].values())
        modes  = dict(image_stats[s]["modes"])
        exts   = dict(image_stats[s]["exts"])
        sizes  = image_stats[s]["sizes"]
        print(f"  {s}:")
        print(f"     modes       : {modes}")
        print(f"     extensions  : {exts}")
        print(f"     unique sizes: {len(sizes)}")
        if sizes:
            ex = sorted(sizes)[0]
            print(f"     example size: {ex[0]} x {ex[1]}")
    print("=" * 78)

In [4]:
# ─── 4a. KL Grading dataset ────────────────────────────────────────────────
kl_counts, kl_stats = count_dataset(KL_ROOT)
print_table("KL GRADING — kneeKL224 (5 classes)", kl_counts, kl_stats)


 KL GRADING — kneeKL224 (5 classes)

[1] Split sizes
  split      images   % of total
  --------------------------------
  train        5778       69.95%
  val           826       10.00%
  test         1656       20.05%
  --------------------------------
  TOTAL        8260

[2] Split × Class matrix (images)
  split         g0      g1      g2      g3      g4   total
  --------------------------------------------------------
  train       2286    1046    1516     757     173    5778
  val          328     153     212     106      27     826
  test         639     296     447     223      51    1656
  --------------------------------------------------------
  TOTAL       3253    1495    2175    1086     251    8260

[3] Class labels and overall distribution
  grade  label         count   % of total
  ----------------------------------------
  g0     Healthy        3253       39.38%
  g1     Doubtful       1495       18.10%
  g2     Minimal        2175       26.33%
  g3     Moderate     

In [5]:
# ─── 4b. YOLOv8 Knee ROI dataset ───────────────────────────────────────────
yolo_counts, yolo_stats = count_dataset(YOLO_ROOT)
print_table("YOLOv8 KNEE ROI — densenet121_yolo_square_roi_trainvaltest_v2", yolo_counts, yolo_stats)


 YOLOv8 KNEE ROI — densenet121_yolo_square_roi_trainvaltest_v2

[1] Split sizes
  split      images   % of total
  --------------------------------
  train        5778       69.95%
  val           826       10.00%
  test         1656       20.05%
  --------------------------------
  TOTAL        8260

[2] Split × Class matrix (images)
  split         g0      g1      g2      g3      g4   total
  --------------------------------------------------------
  train       2286    1046    1516     757     173    5778
  val          328     153     212     106      27     826
  test         639     296     447     223      51    1656
  --------------------------------------------------------
  TOTAL       3253    1495    2175    1086     251    8260

[3] Class labels and overall distribution
  grade  label         count   % of total
  ----------------------------------------
  g0     Healthy        3253       39.38%
  g1     Doubtful       1495       18.10%
  g2     Minimal        2175       26

In [6]:
# ─── 5. Combined summary ───────────────────────────────────────────────────
kl_total   = sum(sum(c.values()) for c in kl_counts.values())
yolo_total = sum(sum(c.values()) for c in yolo_counts.values())

print()
print("=" * 78)
print(" COMBINED SUMMARY")
print("=" * 78)
print(f"  KL Grading images    : {kl_total}")
print(f"  YOLO ROI images      : {yolo_total}")
print(f"  YOLO : KL ratio      : {yolo_total / max(kl_total, 1):.2f}x  (2x = bilateral)")
print("=" * 78)


 COMBINED SUMMARY
  KL Grading images    : 8260
  YOLO ROI images      : 8260
  YOLO : KL ratio      : 1.00x  (2x = bilateral)
